# Ejercicio 25: Web Scraping y Almacenamiento en MongoDB

Este ejercicio introduce a los estudiantes en la extracción de datos de la web mediante web scraping y su almacenamiento en una base de datos NoSQL (MongoDB).

## Objetivos

- Realizar web scraping para extraer datos de un sitio web de noticias.
- Convertir los datos extraídos en formato JSON.
- Almacenar los datos en una base de datos MongoDB.
- Realizar consultas en MongoDB para verificar los datos almacenados.

## Entrada de Datos

1. URL de un sitio web de noticias, por ejemplo, "https://www.bbc.com/news".
2. Datos extraídos: títulos y enlaces de artículos.

## Código Base

### Requerimientos Previos

Antes de comenzar, instale las siguientes dependencias:

```bash
pip install requests beautifulsoup4 pymongo
```

Asegúrese de tener MongoDB instalado y ejecutándose en localhost. En sistemas Ubuntu/Debian, puede instalarlo con:

```bash
sudo apt update
sudo apt install -y mongodb
sudo systemctl start mongodb
```

Para verificar que MongoDB está funcionando:

```bash
mongo --eval "db.runCommand({ ping: 1 })"
```

### 1. Obtener y Analizar Datos de una Página Web


In [ ]:
import requests
from bs4 import BeautifulSoup

# URL de ejemplo
url = "https://www.bbc.com/news"

# Obtener HTML de la página
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Extraer títulos y enlaces de noticias
noticias = []
for articulo in soup.find_all("a", class_="gs-c-promo-heading"):
    titulo = articulo.get_text(strip=True)
    enlace = articulo["href"]
    if not enlace.startswith("http"):
        enlace = "https://www.bbc.com" + enlace
    noticias.append({"titulo": titulo, "enlace": enlace})

# Mostrar algunas noticias extraídas
print(noticias[:5])


### 2. Guardar los Datos en un Archivo JSON


In [ ]:
import json

# Guardar en un archivo JSON
with open("noticias.json", "w", encoding="utf-8") as file:
    json.dump(noticias, file, indent=4, ensure_ascii=False)

print("Noticias guardadas en noticias.json")


### 3. Insertar Datos en MongoDB


In [ ]:
from pymongo import MongoClient

# Conectar a MongoDB
client = MongoClient("mongodb://localhost:27017/")

# Crear base de datos y colección
db = client["web_scraping"]
collection = db["noticias"]

# Cargar el JSON y guardarlo en MongoDB
with open("noticias.json", "r", encoding="utf-8") as file:
    data = json.load(file)
    collection.insert_many(data)

print("Noticias insertadas en MongoDB")


### 4. Consultar la Base de Datos


In [ ]:
# Ver cantidad de noticias almacenadas
print(f"Total de noticias almacenadas: {collection.count_documents({})}")

# Mostrar 5 documentos sin el campo HTML
for noticia in collection.find({}, {"_id": 0, "titulo": 1, "enlace": 1}).limit(5):
    print(noticia)


## Reflexión Final

1. ¿Qué dificultades encontraste al extraer datos del HTML?
2. ¿Cómo podrías mejorar la estructura del JSON almacenado?
3. ¿Cómo manejarías un scraping con múltiples páginas?
4. ¿En qué casos sería más útil MongoDB que una base relacional como PostgreSQL?

## Entrega Esperada

Un script en Python (`.py`) que:

- Realice scraping de un sitio de noticias.
- Guarde los datos en un JSON.
- Inserte el JSON en una colección de MongoDB.
- Ejecute consultas para verificar la información almacenada.